# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [29]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm
from itertools import product
import warnings
warnings.filterwarnings('ignore')

dayofweek_not_scaled = "../../datasets/day-of-week-not-scaled.csv"
dayofweek = "../../datasets/dayofweek.csv"

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [30]:
df = pd.read_csv(dayofweek_not_scaled)
df.head()

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [31]:
df2 = pd.read_csv(dayofweek)
df["dayofweek"] = df2["dayofweek"]


In [32]:
X = df.drop(columns="dayofweek")
y = df["dayofweek"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [33]:
kernel = ["linear", "rbf", "sigmoid"]
C = [0.01, 0.1, 1, 1.5, 5, 10]
gamma = ["scale", "auto"]
class_weight = ["balanced", None]
svc = SVC(random_state=21, probability=True)
param = {"kernel":kernel, 
         "C":C, 
         "gamma": gamma, 
         "class_weight": class_weight}
model_grid = GridSearchCV(svc, param_grid=param, scoring="accuracy", n_jobs=-1)
model_grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC(probabili...ndom_state=21)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'class_weight': ['balanced', None], 'gamma': ['scale', 'auto'], 'kernel': ['linear', 'rbf', ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,

In [34]:
svc_result = pd.DataFrame(model_grid.cv_results_)
svc_result.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_class_weight,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.396345,0.053226,0.018777,0.012255,0.01,balanced,scale,linear,"{'C': 0.01, 'class_weight': 'balanced', 'gamma': 'scale', 'kernel': 'linear'}",0.325926,0.344444,0.337037,0.345725,0.345725,0.339771,0.007641,36
1,0.479992,0.017214,0.021446,0.005595,0.01,balanced,scale,rbf,"{'C': 0.01, 'class_weight': 'balanced', 'gamma': 'scale', 'kernel': 'rbf'}",0.233333,0.233333,0.233333,0.059480,0.237918,0.199480,0.070023,59
2,0.608462,0.087376,0.019745,0.007458,0.01,balanced,scale,sigmoid,"{'C': 0.01, 'class_weight': 'balanced', 'gamma': 'scale', 'kernel': 'sigmoid'}",0.233333,0.233333,0.233333,0.059480,0.237918,0.199480,0.070023,59
3,0.362188,0.016124,0.010662,0.000907,0.01,balanced,auto,linear,"{'C': 0.01, 'class_weight': 'balanced', 'gamma': 'auto', 'kernel': 'linear'}",0.325926,0.344444,0.337037,0.345725,0.345725,0.339771,0.007641,36
4,0.464864,0.056750,0.017944,0.001622,0.01,balanced,auto,rbf,"{'C': 0.01, 'class_weight': 'balanced', 'gamma': 'auto', 'kernel': 'rbf'}",0.233333,0.233333,0.233333,0.059480,0.237918,0.199480,0.070023,59


In [35]:
model_grid.best_params_

{'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}

## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [36]:
depth = list(range(1,50))
class_weight = ["balanced", None]
criterion = ["entropy", "gini"]
tree = DecisionTreeClassifier(random_state=21)
param = {"max_depth": depth, 
         "class_weight": class_weight, 
         "criterion":criterion}
model_tree = GridSearchCV(tree, param_grid=param, scoring="accuracy", n_jobs=-1)
model_tree.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeC...ndom_state=21)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'class_weight': ['balanced', None], 'criterion': ['entropy', 'gini'], 'max_depth': [1, 2, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: int, de

In [37]:
model_tree_result = pd.DataFrame(model_tree.cv_results_)
model_tree_result_sorted = model_tree_result.sort_values("rank_test_score")



In [38]:
model_tree.best_params_

{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 23}

## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [39]:
estimators = [5,10, 50, 100]
depth = list(range(1,50))
class_weight = ["balanced", None]
criterion = ["entropy", "gini"]
forest = RandomForestClassifier(random_state=21)
param = {"max_depth": depth,
         "class_weight":class_weight , 
         "criterion": criterion, 
         "n_estimators": estimators}
model_forest = GridSearchCV(forest, param_grid=param, scoring="accuracy", n_jobs=-1)
model_forest.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=21)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'class_weight': ['balanced', None], 'criterion': ['entropy', 'gini'], 'max_depth': [1, 2, ...], 'n_estimators': [5, 10, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",N

In [40]:
forest_result = pd.DataFrame(model_forest.cv_results_)
forest_result.sort_values("rank_test_score")
forest_result.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_class_weight,param_criterion,param_max_depth,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.010304,0.001293,0.002736,0.000541,balanced,entropy,1,5,"{'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 1, 'n_estimators': 5}",0.274074,0.251852,0.292593,0.289963,0.330855,0.287867,0.025931,783
1,0.015357,0.002027,0.002990,0.000723,balanced,entropy,1,10,"{'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 1, 'n_estimators': 10}",0.281481,0.362963,0.403704,0.386617,0.330855,0.353124,0.043372,780
2,0.062066,0.006466,0.005215,0.000922,balanced,entropy,1,50,"{'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 1, 'n_estimators': 50}",0.462963,0.462963,0.459259,0.457249,0.446097,0.457706,0.006208,756
3,0.123999,0.007907,0.008048,0.000961,balanced,entropy,1,100,"{'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 1, 'n_estimators': 100}",0.474074,0.462963,0.429630,0.434944,0.460967,0.452515,0.017192,759
4,0.010642,0.000777,0.002855,0.000166,balanced,entropy,2,5,"{'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 2, 'n_estimators': 5}",0.311111,0.307407,0.351852,0.282528,0.304833,0.311546,0.022490,782


In [41]:
model_forest.best_params_

{'class_weight': None,
 'criterion': 'gini',
 'max_depth': 28,
 'n_estimators': 50}

## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [42]:
def func(model, param_model, scoring, cv = 5, n_jobs = -1):
    keys = list(param_model.keys())
    value = list(product(*param_model.values()))
    result = []

    for values in tqdm(value):
        param = dict(zip(keys, values))
        c_m = model.set_params(**param)
        score = cross_val_score(c_m, X_train, y_train, scoring = scoring, cv = cv, n_jobs = n_jobs)
        result.append({**param, "mean_accuracy": score.mean(), "std_accuracy": score.std()})
    return pd.DataFrame(result)

In [43]:
estimators = [5,10, 50, 100]
depth = list(range(1,50))
class_weight = ["balanced", None]
criterion = ["entropy", "gini"]
forest = RandomForestClassifier(random_state=21)
param = {"max_depth": depth,
         "class_weight":class_weight , 
         "criterion": criterion, 
         "n_estimators": estimators}
model_forest = GridSearchCV(forest, param_grid=param, scoring="accuracy", n_jobs=-1)
func_res = func(forest, param, "accuracy")

  0%|          | 0/784 [00:00<?, ?it/s]

In [44]:
func_res.sort_values("mean_accuracy", ascending=False, inplace=True)
func_res.head()

,max_depth,class_weight,criterion,n_estimators,mean_accuracy,std_accuracy
446,28,NaN,gini,50,0.904290,0.010961
495,31,NaN,gini,100,0.903547,0.014380
751,47,NaN,gini,100,0.902806,0.010460
767,48,NaN,gini,100,0.902806,0.010460
703,44,NaN,gini,100,0.902806,0.010460


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [45]:
best_model = RandomForestClassifier(max_depth = 24, class_weight="balanced", criterion="entropy", n_estimators=100, random_state=21)

In [46]:
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

In [47]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.9289940828402367